# Milestone 3 — CNN from Scratch


In [ ]:
!pip install wandb -q

In [ ]:
import os, random, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CONFIG

SR, DURATION = 22050, 5
SAMPLES      = SR * DURATION         

N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
FMIN, FMAX   = 20, 8000
SPEC_W       = (SAMPLES // HOP_LENGTH) + 1   

BATCH_SIZE   = 64
NUM_EPOCHS   = 30
LR           = 1e-3
WD           = 1e-4
DROPOUT      = 0.4
NUM_CLASSES  = 10
NUM_WORKERS  = 2


N_AUG        = 6    
N_VAL        = 30   
NOISE_MIN    = 0.05
NOISE_MAX    = 0.35
N_TTA        = 10   

GENRES = sorted(['blues','classical','country','disco','hiphop',
                 'jazz','metal','pop','reggae','rock'])

G2I = {g:i for i,g in enumerate(GENRES)}
I2G = {i:g for g,i in G2I.items()}

#Path
BASE = None
for p in ['/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup',
          '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup']:
    if os.path.exists(p): BASE = p; break
assert BASE, 'Dataset not found!'

GENRES_PATH  = f'{BASE}/genres_stems'
NOISE_PATH   = f'{BASE}/ESC-50-master/audio'
MASHUPS_PATH = f'{BASE}/mashups'
TEST_CSV     = f'{BASE}/test.csv'
print(f'Dataset: {BASE}')
print(f'Spec shape: {N_MELS} × {SPEC_W}')

In [ ]:
# BUILD CATALOG

catalog = {}
for g in GENRES:
    gdir = f'{GENRES_PATH}/{g}'
    songs = sorted([f'{gdir}/{s}' for s in os.listdir(gdir)
                    if os.path.isdir(f'{gdir}/{s}')])
    catalog[g] = songs
    print(f'  {g:12s}: {len(songs)} songs')

# Detect actual stem names
STEMS = [f for f in os.listdir(catalog[GENRES[0]][0]) if f.endswith('.wav')]
print(f'Stems: {STEMS}')

noise_files = [f'{NOISE_PATH}/{f}' for f in os.listdir(NOISE_PATH) if f.endswith('.wav')]
print(f'Noise files: {len(noise_files)}')

# Train/val split BY SONGS (prevent data leakage)
rng = random.Random(SEED)
tr_songs, vl_songs = {}, {}
for g, songs in catalog.items():
    s = songs.copy(); rng.shuffle(s)
    n = max(2, int(len(s)*0.15))
    vl_songs[g] = s[:n]; tr_songs[g] = s[n:]
print('Split:', {g: f'{len(tr_songs[g])}tr/{len(vl_songs[g])}vl' for g in GENRES})

In [ ]:
# AUDIO HELPERS


def load_stem(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True, duration=DURATION+2)
        if len(y) < SAMPLES:
            y = np.pad(y, (0, SAMPLES-len(y)))
        else:
            s = random.randint(0, len(y)-SAMPLES)
            y = y[s:s+SAMPLES]
        return y.astype(np.float32)
    except:
        return np.zeros(SAMPLES, np.float32)

def load_noise():
    try:
        y, _ = librosa.load(random.choice(noise_files), sr=SR, mono=True)
        if len(y) < SAMPLES:
            y = np.tile(y, (SAMPLES//len(y))+1)
        s = random.randint(0, len(y)-SAMPLES)
        return y[s:s+SAMPLES].astype(np.float32)
    except:
        return np.zeros(SAMPLES, np.float32)


def make_mashup(genre, song_pool):

    # Mix stems from 4 DIFFERENT songs of same genre + ESC-50 noise.
    
    songs = random.choices(song_pool[genre], k=4)
    stems = STEMS.copy()
    random.shuffle(stems)
    mixed = np.zeros(SAMPLES, np.float32)
    for song, stem in zip(songs, stems[:4]):
        y = load_stem(f'{song}/{stem}')
        mx = np.max(np.abs(y))
        if mx > 0: y /= (mx + 1e-6)
        mixed += random.uniform(0.4, 1.0) * y
    mx = np.max(np.abs(mixed))
    if mx > 0: mixed /= (mx + 1e-6)
    #Adding noise
    noise  = load_noise()
    mixed += random.uniform(NOISE_MIN, NOISE_MAX) * noise
    mx = np.max(np.abs(mixed))
    if mx > 0: mixed /= (mx + 1e-6)
    return mixed


def to_spec(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
    mel = librosa.power_to_db(mel, ref=np.max)

    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-9)
    if mel.shape[1] < SPEC_W:
        mel = np.pad(mel, ((0,0),(0, SPEC_W-mel.shape[1])))
    return mel[:, :SPEC_W].astype(np.float32)


print('Audio helpers ready.')
print(f'Testing make_mashup...')
test_audio = make_mashup('rock', tr_songs)
test_spec  = to_spec(test_audio)
print(f'Audio shape: {test_audio.shape} | Spec shape: {test_spec.shape}  ✅')

In [ ]:
# VISUALIZE — what we're feeding the CNN

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle('Mel-Spectrograms of Simulated Training Mashups', fontsize=13, fontweight='bold')
for ax, genre in zip(axes.flat, GENRES):
    spec = to_spec(make_mashup(genre, tr_songs))
    ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(genre.upper(), fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Mel Freq')
plt.tight_layout()
plt.savefig('mel_spectrograms.png', dpi=100)
plt.show()
print('Notice different visual patterns per genre...this is what the CNN learns!')

In [ ]:
# PRE-COMPUTE ALL SPECTROGRAMS
# Pre-compute everything ONCE → store in RAM.
# Training then reads from RAM (near-instant per batch).

n_est = sum(len(v) for v in tr_songs.values()) * N_AUG
print(f'Pre-computing ~{n_est} train + {len(GENRES)*N_VAL} val spectrograms...')
print('(Runs once — then training is fast)')

X_tr, y_tr = [], []
X_vl, y_vl = [], []
t0 = time.time()

for g in GENRES:
    lbl = G2I[g]
    # Training: N_AUG mashups per song, each one different
    for _ in tqdm(range(len(tr_songs[g]) * N_AUG), desc=f'TR {g}', leave=False):
        X_tr.append(to_spec(make_mashup(g, tr_songs)))
        y_tr.append(lbl)
    # Validation: fixed set generated once
    for _ in tqdm(range(N_VAL), desc=f'VL {g}', leave=False):
        X_vl.append(to_spec(make_mashup(g, vl_songs)))
        y_vl.append(lbl)

X_tr = np.array(X_tr, np.float32)
y_tr = np.array(y_tr, np.int64)
X_vl = np.array(X_vl, np.float32)
y_vl = np.array(y_vl, np.int64)

print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'X_tr: {X_tr.shape}  ({X_tr.nbytes/1e6:.0f} MB)')
print(f'X_vl: {X_vl.shape}  ({X_vl.nbytes/1e6:.0f} MB)')
print(f'Train labels: {np.bincount(y_tr)}')
print(f'Val   labels: {np.bincount(y_vl)}')

In [ ]:
# DATASET & DATALOADERS

class SpecDS(Dataset):
    
    def __init__(self, X, y, aug=False):
        self.X   = torch.tensor(X).unsqueeze(1) 
        self.y   = torch.tensor(y)
        self.aug = aug
        # SpecAugment: randomly mask frequency and time bands
        self.fm  = T.FrequencyMasking(freq_mask_param=20)
        self.tm  = T.TimeMasking(time_mask_param=40)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.aug:
            if random.random() > 0.4: x = self.fm(x)
            if random.random() > 0.4: x = self.tm(x)
        return x, self.y[i]

tr_ds = SpecDS(X_tr, y_tr, aug=True)
vl_ds = SpecDS(X_vl, y_vl, aug=False)

tr_dl = DataLoader(tr_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS,
                   pin_memory=True, drop_last=True)
vl_dl = DataLoader(vl_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                   pin_memory=True)

print(f'Train: {len(tr_ds)} samples | {len(tr_dl)} batches')
print(f'Val:   {len(vl_ds)} samples | {len(vl_dl)} batches')
xb, yb = next(iter(tr_dl))
print(f'Batch shape: {xb.shape}  ← (batch, channels, freq_bins, time_frames)')

In [ ]:
# CNN MODEL — Built from scratch

class ConvBlock(nn.Module):
    """Conv → BN → ReLU → Conv → BN → ReLU → MaxPool"""
    def __init__(self, ic, oc, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(ic, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(True),
            nn.Conv2d(oc, oc, 3, padding=1, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(True),
        ]
        if pool: layers.append(nn.MaxPool2d(2, 2))
        self.b = nn.Sequential(*layers)
    def forward(self, x): return self.b(x)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: (B, 1, 128, 216)
        self.features = nn.Sequential(
            ConvBlock(1,   32,  pool=True),   # → (B, 32,  64, 108)
            ConvBlock(32,  64,  pool=True),   # → (B, 64,  32,  54)
            ConvBlock(64,  128, pool=True),   # → (B, 128, 16,  27)
            ConvBlock(128, 256, pool=True),   # → (B, 256,  8,  13)
            ConvBlock(256, 256, pool=False),  # → (B, 256,  8,  13)
        )

        self.gap  = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Dropout(DROPOUT/2),
            nn.Linear(128, NUM_CLASSES),
        )
        # He initialization for ReLU networks
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)   
        x = self.gap(x)        
        return self.head(x)    


model = CNN().to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')
with torch.no_grad():
    o = model(torch.randn(2, 1, N_MELS, SPEC_W).to(DEVICE))
    print(f'Output shape: {o.shape}  ✅')

In [ ]:
# LOSS / OPTIMIZER / SCHEDULER

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print('Loss:      CrossEntropyLoss (label_smoothing=0.1)')
print('Optimizer: AdamW')
print('Scheduler: CosineAnnealingLR')

In [ ]:
# # W&B INIT

# wandb.login()  

# run = wandb.init(
#     project='genre-classification',
#     name='milestone3-scratch-cnn',
#     config=dict(
#         n_mels=N_MELS, hop=HOP_LENGTH, sr=SR, duration=DURATION,
#         batch_size=BATCH_SIZE, epochs=NUM_EPOCHS, lr=LR, wd=WD,
#         dropout=DROPOUT, n_aug=N_AUG, n_val=N_VAL, n_tta=N_TTA,
#         arch='CNN-5ConvBlock-GAP'
#     ),
#     tags=['milestone-3', 'cnn', 'from-scratch', 'fixed']
# )
# print(run.url)

In [ ]:
# TRAINING LOOP

def train_ep(model, dl):
    model.train()
    loss_sum, ps, ls = 0.0, [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item()
        ps.extend(out.argmax(1).cpu().numpy())
        ls.extend(y.cpu().numpy())
    return loss_sum/len(dl), f1_score(ls, ps, average='macro')

@torch.no_grad()
def val_ep(model, dl):
    model.eval()
    loss_sum, ps, ls = 0.0, [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        out = model(x)
        loss_sum += criterion(out, y).item()
        ps.extend(out.argmax(1).cpu().numpy())
        ls.extend(y.cpu().numpy())
    return loss_sum/len(dl), f1_score(ls, ps, average='macro'), ps, ls


best_f1, best_ep = 0.0, 0
hist = {k: [] for k in ['tl','tf','vl','vf']}

print('🚀 Training!')
print('='*65)
t_all = time.time()

for ep in range(NUM_EPOCHS):
    t0 = time.time()
    tl, tf = train_ep(model, tr_dl)
    vl, vf, _, _ = val_ep(model, vl_dl)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    for k,v in zip(['tl','tf','vl','vf'],[tl,tf,vl,vf]):
        hist[k].append(v)
    # wandb.log({'ep':ep+1,'tr/loss':tl,'tr/f1':tf,'val/loss':vl,'val/f1':vf,'lr':lr})

    flag = ''
    if vf > best_f1:
        best_f1, best_ep = vf, ep+1
        torch.save({'ep':ep+1,'state':model.state_dict(),'f1':vf}, 'best_cnn_m3.pth')
        flag = ' ✨'

    print(f'Ep {ep+1:02d}/{NUM_EPOCHS} | '
          f'Tr {tl:.4f}/{tf:.4f} | '
          f'Vl {vl:.4f}/{vf:.4f} | '
          f'{time.time()-t0:.0f}s{flag}')

print(f'\n Done in {(time.time()-t_all)/60:.1f} min')
print(f'🏆 Best Val F1: {best_f1:.4f} (epoch {best_ep})')

In [ ]:
# TRAINING CURVES

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep_r = range(1, NUM_EPOCHS+1)
axes[0].plot(ep_r, hist['tl'], label='Train'); axes[0].plot(ep_r, hist['vl'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep_r, hist['tf'], label='Train'); axes[1].plot(ep_r, hist['vf'], label='Val')
axes[1].axhline(best_f1, color='green', ls='--', label=f'Best {best_f1:.4f}')
axes[1].set_title('Macro F1'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle(f'CNN Training — Best Val F1: {best_f1:.4f}', fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_curves.png', dpi=100); plt.show()
# wandb.log({'curves': wandb.Image('cnn_curves.png')})

In [ ]:
# CONFUSION MATRIX + PER-CLASS REPORT

ckpt = torch.load('best_cnn_m3.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state'])
print(f'Best model: epoch {ckpt["ep"]}, val F1={ckpt["f1"]:.4f}')

_, vf, vp, vl_true = val_ep(model, vl_dl)
print(f'\nFinal Val Macro F1: {vf:.4f}')
print('\nPer-Class Report:')
print(classification_report(vl_true, vp, target_names=GENRES, digits=3))

cm = confusion_matrix(vl_true, vp)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=GENRES, yticklabels=GENRES, cmap='Blues')
plt.title(f'Confusion Matrix (Val F1={vf:.4f})')
plt.xticks(rotation=45); plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.savefig('cnn_cm.png', dpi=100); plt.show()
# wandb.log({'cm': wandb.Image('cnn_cm.png'), 'best_val_f1': vf})

In [ ]:
# INFERENCE WITH TTA

test_df = pd.read_csv(TEST_CSV)
print(f'Test: {len(test_df)} samples')

lookup = {}
for f in os.listdir(MASHUPS_PATH):
    if f.endswith('.wav'):
        num = int(f.replace('song','').replace('.wav',''))
        lookup[num] = f'{MASHUPS_PATH}/{f}'
print(f'Lookup: {len(lookup)} files')
print(f'Matched: {sum(1 for i in test_df["id"] if int(i) in lookup)}/{len(test_df)}')

model.eval()
all_ids, all_preds = [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f'Inference TTA×{N_TTA}'):
    sid = int(row['id'])
    fp  = lookup.get(sid)
    if not fp:
        all_ids.append(row['id']); all_preds.append('pop'); continue
    try:
        audio, _ = librosa.load(fp, sr=SR, mono=True)
    except:
        all_ids.append(row['id']); all_preds.append('pop'); continue

    probs = np.zeros(NUM_CLASSES)
    for _ in range(N_TTA):
        if len(audio) <= SAMPLES:
            seg = np.pad(audio, (0, SAMPLES-len(audio)))
        else:
            s   = random.randint(0, len(audio)-SAMPLES)
            seg = audio[s:s+SAMPLES]
        t = torch.tensor(to_spec(seg.astype(np.float32))).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs += F.softmax(model(t), 1).cpu().numpy()[0]

    all_ids.append(row['id'])
    all_preds.append(I2G[np.argmax(probs)])

print(f'Done! {len(all_preds)} predictions')

In [ ]:
# SUBMISSION

sub = pd.DataFrame({'id': all_ids, 'genre': all_preds})
assert all(g in set(GENRES) for g in sub['genre']), 'Invalid genre found!'
assert len(sub) == len(test_df), 'Row count mismatch!'

sub.to_csv('submission.csv', index=False)
print('Prediction distribution:')
print(sub['genre'].value_counts())
print('\n Saved: submission.csv')

# wandb.summary.update({'best_val_f1': best_f1, 'best_epoch': best_ep})
# wandb.finish()
print(f'\n🏆 Best Val F1: {best_f1:.4f}')